# 05 — Ladder summary

All 9 rungs side-by-side. The single chart that tells the F3 + F2 story.


In [ ]:
import sys; from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src import config, eval as evalmod


In [ ]:
RUNG_IDS = ['rung_1_simple_momentum_decile', 'rung_1d_simple_momentum_dual',
            'rung_2_ewm_momentum_decile',    'rung_2d_ewm_momentum_dual',
            'rung_3_ts_regression_decile',   'rung_3d_ts_regression_dual',
            'rung_4_linear_tcnn',            'rung_5_tcnn_1ch', 'rung_6_tcnn_3ch']
master = evalmod.load_master_results(RUNG_IDS)
print(f'{len(master):,} daily-return records across {master["experiment_id"].nunique()} rungs')


## Sharpe / max-DD table (the headline)


In [ ]:
summary = (master.groupby('experiment_id')['return']
                  .apply(lambda s: pd.Series(evalmod.perf_summary(s)))
                  .unstack())
summary = summary[['ann_return', 'ann_vol', 'sharpe', 'max_dd', 't_stat', 'n_days']]
summary.loc[RUNG_IDS].round(3)  # ordered by rung


## Cumulative P&L plot (cumsum)


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
for exp_id in RUNG_IDS:
    df = master[master['experiment_id'] == exp_id].sort_values('date')
    ax.plot(df['date'], df['return'].cumsum(), label=exp_id, alpha=0.85)
ax.set(title='Comparison ladder — cumulative P&L (cumsum)', ylabel='cum return')
ax.legend(loc='upper left', fontsize=8); ax.grid(alpha=0.3); plt.tight_layout()


## Decomposition: F3 (learnable aggregation) and F2 (end-to-end) gaps


In [ ]:
# F3 ladder: 1 → 2 → 3 → 4 → 5 (each step adds one capability)
f3_ladder = ['rung_1_simple_momentum_decile', 'rung_2_ewm_momentum_decile',
             'rung_3_ts_regression_decile', 'rung_4_linear_tcnn', 'rung_5_tcnn_1ch']
f3 = summary.loc[f3_ladder, ['sharpe']].copy()
f3['delta_sharpe'] = f3['sharpe'].diff()
f3.round(3)
